[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/46_sft_data_solution.ipynb)

#  Solution: SFT Data Construction

Reference solution for building supervised fine-tuning datasets  the core preprocessing step in instruction tuning (Alpaca, Vicuna, LLaMA-chat, etc.).

```text
1. Concatenate instruction + response
2. Tokenize full text and instruction-only
3. Mask instruction tokens as -100 in labels
4. Pad/truncate to max_length
5. Stack into batched tensors
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
#  SOLUTION

def build_sft_dataset(conversations, tokenizer, max_length=2048):
    """Build SFT dataset from conversation data.
    
    Inspired by HuggingFace TRL's SFTTrainer, LLaMA-Factory, and Axolotl.
    Key design: labels mask instruction tokens (-100) so loss only
    backpropagates through the assistant response.
    """
    pad_id = getattr(tokenizer, 'pad_token_id', 0)
    all_input_ids = []
    all_labels = []
    
    for conv in conversations:
        instruction = conv["instruction"]
        response = conv["response"]
        
        # Tokenize full concatenated text
        full_text = instruction + " " + response
        full = tokenizer(full_text, max_length=max_length, truncation=True)
        full_ids = full["input_ids"]
        
        # Tokenize instruction only to find split boundary
        inst = tokenizer(instruction, max_length=max_length, truncation=True)
        inst_ids = inst["input_ids"]
        
        # Determine instruction token count
        if 0 < len(inst_ids) < len(full_ids):
            split_point = len(inst_ids)
        else:
            # Fallback: proportional split by character length
            ratio = len(instruction) / max(1, len(full_text))
            split_point = max(1, round(len(full_ids) * ratio))
        
        # Create labels with instruction masked
        labels = full_ids.clone().to(dtype=torch.long)
        labels[:split_point] = -100
        
        # Pad or truncate to max_length
        if len(full_ids) < max_length:
            pad_len = max_length - len(full_ids)
            full_ids = torch.cat([full_ids, torch.full((pad_len,), pad_id, dtype=full_ids.dtype)])
            labels = torch.cat([labels, torch.full((pad_len,), -100, dtype=labels.dtype)])
        elif len(full_ids) > max_length:
            full_ids = full_ids[:max_length]
            labels = labels[:max_length]
        
        all_input_ids.append(full_ids)
        all_labels.append(labels)
    
    return {
        "input_ids": torch.stack(all_input_ids),
        "labels": torch.stack(all_labels),
    }

In [ ]:
#  Verify

class SimpleTokenizer:
    def __init__(self):
        self.pad_token_id = 0
    def __call__(self, text, max_length=None, truncation=False):
        ids = [ord(c) for c in text] + [2]
        if max_length and len(ids) > max_length:
            ids = ids[:max_length-1] + [2]
        return {"input_ids": torch.tensor(ids)}

convs = [{"instruction": "Hello", "response": "World"}]
result = build_sft_dataset(convs, SimpleTokenizer(), max_length=32)
print("input_ids:", result["input_ids"])
print("labels:   ", result["labels"])
# Instruction region should be -100
assert (result["labels"][0, :3] == -100).all()

In [ ]:
# Run judge
from torch_judge import check
check("sft_data")